In [5]:
with open('../MCA-net/ptbdb/Fold') as fp:
    lines = fp.readlines()
# with open('F:\pythonProject\代码转接\model\MCA-net\ptbdb\Fold') as fp:
#     lines = fp.readlines()

In [2]:
import os
print(os.getcwd())

F:\pythonProject\代码转接\model\MCA-net


In [1]:
import timm

In [4]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import timm

# Define a custom dataset
class CustomDataset(Dataset):
    def __init__(self, size=1000, input_size=(3, 224, 224), num_classes=10):
        self.data = torch.randn(size, *input_size)
        self.targets = torch.randint(0, num_classes, (size,))
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx], self.targets[idx]

# Create custom dataset and dataloaders
train_dataset = CustomDataset(size=1000)
test_dataset = CustomDataset(size=200)

train_loader = DataLoader(dataset=train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=32, shuffle=False)

# Build model
model = timm.create_model('resnet18', pretrained=False)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 10)  # Assuming 10 classes

# Define device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Define loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 5
for epoch in range(num_epochs):
    for i, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.to(device)
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        if (i+1) % 10 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(train_loader)}], Loss: {loss.item():.4f}')

# Testing
model.eval()
with torch.no_grad():
    correct = 0
    total = 0
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print(f'Accuracy of the model on the test images: {100 * correct / total}%')

Epoch [1/5], Step [10/32], Loss: 2.2998
Epoch [1/5], Step [20/32], Loss: 2.4821


KeyboardInterrupt: 